<a href="https://colab.research.google.com/github/Soviet117/lima-rent-predictive/blob/main/notebooks/01_ETL_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ============================================
# BLOQUE 1: CONFIGURACIÓN INICIAL
# ============================================

# 1. Instalar PySpark
!pip install pyspark -q

# 2. Importar librerías
import os
import shutil
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Librerías importadas")

Librerías importadas


In [12]:
# ============================================
# BLOQUE 2: MONTAR DRIVE Y COPIAR ARCHIVO
# ============================================

# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Verificar y copiar archivo
drive_path = '/content/drive/MyDrive/lima_rent/dataset.csv'
local_path = '/tmp/dataset.csv'

if os.path.exists(drive_path):
    print("Archivo encontrado en Drive")
    shutil.copy2(drive_path, local_path)
    print(f"Archivo copiado a: {local_path}")
else:
    print("Archivo NO encontrado en Drive")
    raise FileNotFoundError("El archivo dataset.csv no existe")

# 3. Verificar copia
if os.path.exists(local_path):
    print("Archivo disponible localmente")
    !ls -la /tmp/dataset.csv
else:
    raise FileNotFoundError("Error al copiar el archivo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivo encontrado en Drive
Archivo copiado a: /tmp/dataset.csv
Archivo disponible localmente
-rw------- 1 root root 316275 Jul  1 18:01 /tmp/dataset.csv


In [13]:
# ============================================
# BLOQUE 3: CREAR SESIÓN SPARK
# ============================================

spark = SparkSession.builder \
    .appName("LimaRentETL") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"Sesión Spark creada. Versión: {spark.version}")

Sesión Spark creada. Versión: 4.0.3


In [14]:
# ============================================
# BLOQUE 4: CARGAR DATOS CON SPARK
# ============================================

sdf = spark.read.csv('/tmp/dataset.csv', header=True, inferSchema=True)

print(f"Registros cargados: {sdf.count()}")
print("\nEsquema del dataset:")
sdf.printSchema()
print("\nPrimeras 5 filas:")
sdf.show(5, truncate=True)

Registros cargados: 1082

Esquema del dataset:
root
 |-- title: string (nullable = true)
 |-- location: string (nullable = true)
 |-- price: string (nullable = true)
 |-- bedroom: string (nullable = true)
 |-- bathroom: string (nullable = true)
 |-- area: string (nullable = true)
 |-- year_contruction: string (nullable = true)
 |-- maintenance: string (nullable = true)
 |-- housing_type: string (nullable = true)
 |-- operation_type: string (nullable = true)
 |-- date_pub: string (nullable = true)
 |-- url: string (nullable = true)


Primeras 5 filas:
+--------------------+--------------------+--------+-------------+--------+------+----------------+-----------+------------+--------------+--------------------+--------------------+
|               title|            location|   price|      bedroom|bathroom|  area|year_contruction|maintenance|housing_type|operation_type|            date_pub|                 url|
+--------------------+--------------------+--------+-------------+--------+----

In [33]:
# ============================================
# BLOQUE 5: ETL - LIMPIEZA DE DATOS
# ============================================

print("Iniciando ETL - Limpieza...")

# 1. Renombrar columnas
sdf = sdf.withColumnRenamed("title", "titulo") \
         .withColumnRenamed("location", "ubicacion") \
         .withColumnRenamed("price", "precio") \
         .withColumnRenamed("bedroom", "dormitorios") \
         .withColumnRenamed("bathroom", "banos") \
         .withColumnRenamed("area", "area") \
         .withColumnRenamed("year_contruction", "anio_construccion") \
         .withColumnRenamed("maintenance", "mantenimiento") \
         .withColumnRenamed("housing_type", "tipo_inmueble") \
         .withColumnRenamed("operation_type", "tipo_operacion") \
         .withColumnRenamed("date_pub", "fecha_publicacion") \
         .withColumnRenamed("url", "url")
print("Columnas renombradas")

# 2. Eliminar nulos en columnas clave
sdf = sdf.dropna(subset=["precio", "ubicacion"])
print(f"Registros despues de limpiar nulos: {sdf.count()}")

# 3. Limpiar precio (quitar USD, S/. y comas)
sdf = sdf.withColumn("precio_limpio",
                     regexp_replace(col("precio"), r"[^0-9.]", ""))
sdf = sdf.withColumn("precio_limpio",
                     when(col("precio_limpio") == "", None)
                     .otherwise(col("precio_limpio").cast(DoubleType())))
print("Precio limpiado")

# 4. CORRECCION DEFINITIVA: Limpiar area con regexp_extract
from pyspark.sql.functions import regexp_extract

# Extraer SOLO numeros del campo area (ignorar texto como "1 Medio baño")
sdf = sdf.withColumn("area_limpia_str", regexp_extract(col("area"), r"^(\d+\.?\d*)", 1))

# Convertir a Double, si no se puede extraer numero, queda NULL
sdf = sdf.withColumn("area_limpia",
                     when(col("area_limpia_str") != "",
                          col("area_limpia_str").cast(DoubleType()))
                     .otherwise(None))

# Eliminar registros con area NULL o <= 0
sdf = sdf.filter(col("area_limpia").isNotNull() & (col("area_limpia") > 0))
sdf = sdf.drop("area_limpia_str")
print(f"Area limpiada. Registros con area valida: {sdf.count()}")

# 5. Extraer distrito de manera segura
from pyspark.sql.functions import split, when, size

sdf = sdf.withColumn("ubicacion_split", split(col("ubicacion"), ","))
sdf = sdf.withColumn("distrito",
                     when(size(col("ubicacion_split")) >= 2,
                          trim(col("ubicacion_split").getItem(1)))
                     .otherwise(None))
sdf = sdf.drop("ubicacion_split")
print("Distrito extraido de manera segura")

# 6. Limpiar dormitorios
sdf = sdf.withColumn("dormitorios_limpio",
                     regexp_extract(col("dormitorios"), r"(\d+)", 1).cast(IntegerType()))
print("Dormitorios limpiados")

# 7. Limpiar banos (extraer SOLO el primer numero)
sdf = sdf.withColumn("banos_limpio",
                     regexp_extract(col("banos"), r"(\d+)", 1).cast(IntegerType()))
print("Banos limpiados")

# 8. CORRECCION: Arreglar precios en Soles
sdf = sdf.withColumn("es_usd", when(col("precio").contains("USD"), True).otherwise(False))
sdf = sdf.withColumn("es_soles", when(col("precio").contains("S/."), True).otherwise(False))

sdf = sdf.withColumn("precio_usd",
                     when(col("es_soles") == True, col("precio_limpio") / 3.8)
                     .when(col("es_usd") == True, col("precio_limpio"))
                     .otherwise(col("precio_limpio")))

sdf = sdf.withColumn("precio_usd",
                     when(col("precio_usd").isNotNull(),
                          round(col("precio_usd"), 2))
                     .otherwise(None))
print("Precios estandarizados a USD")

# 9. Eliminar registros con precios sospechosos
sdf = sdf.filter(col("precio_usd") > 1)
print(f"Registros despues de filtrar precios < 1 USD: {sdf.count()}")

# 10. Mostrar verificacion
print("\nVerificacion de datos limpios:")
sdf.select("precio", "precio_usd", "dormitorios_limpio", "banos_limpio",
           "area_limpia", "distrito", "tipo_inmueble").show(10)

Iniciando ETL - Limpieza...
Columnas renombradas
Registros despues de limpiar nulos: 598
Precio limpiado
Area limpiada. Registros con area valida: 597
Distrito extraido de manera segura
Dormitorios limpiados
Banos limpiados
Precios estandarizados a USD
Registros despues de filtrar precios < 1 USD: 597

Verificacion de datos limpios:
+--------+----------+------------------+------------+-----------+----------+-------------+
|  precio|precio_usd|dormitorios_limpio|banos_limpio|area_limpia|  distrito|tipo_inmueble|
+--------+----------+------------------+------------+-----------+----------+-------------+
|  USD900|     900.0|                 2|           2|      103.0|Miraflores|  Apartamento|
|  USD720|     720.0|                 2|           2|       66.0|  Barranco|  Apartamento|
|USD2,300|    2300.0|                 5|           3|      194.0|      Lima|         Casa|
|USD1,200|    1200.0|                 1|           2|      134.0|     Rimac|  Apartamento|
|USD1,450|    1450.0|       

In [34]:
# ============================================
# BLOQUE 6: ET# ============================================
# BLOQUE 5: ETL - LIMPIEZA DE DATOS
# ============================================

print("Iniciando ETL - Limpieza...")

# 1. Renombrar columnas
sdf = sdf.withColumnRenamed("title", "titulo") \
         .withColumnRenamed("location", "ubicacion") \
         .withColumnRenamed("price", "precio") \
…sdf = sdf.filter(col("precio_usd") > 1)
print(f"Registros despues de filtrar precios < 1 USD: {sdf.count()}")

# 10. Mostrar verificacion
print("\nVerificacion de datos limpios:")
sdf.select("precio", "precio_usd", "dormitorios_limpio", "banos_limpio",
           "area_limpia", "distrito", "tipo_inmueble").show(10)L - VARIABLES DERIVADAS
# ============================================

print("Creando variables derivadas...")

# 1. Precio por metro cuadrado
sdf = sdf.withColumn("precio_m2",
                     when(col("area_limpia") > 0,
                          col("precio_usd") / col("area_limpia"))
                     .otherwise(None))

# 2. CORRECCION: Antiguedad del inmueble (con manejo de errores)
# Primero, limpiar anio_construccion para eliminar valores no numericos
sdf = sdf.withColumn("anio_construccion_limpio",
                     regexp_extract(col("anio_construccion"), r"(\d{4})", 1))

sdf = sdf.withColumn("antiguedad",
                     when(col("anio_construccion_limpio") != "",
                          year(current_date()) - col("anio_construccion_limpio").cast(IntegerType()))
                     .otherwise(None))

# Eliminar columna temporal
sdf = sdf.drop("anio_construccion_limpio")

# 3. Categoria de tamaño
sdf = sdf.withColumn("tamano_categoria",
                     when(col("area_limpia") < 50, "Pequeño")
                     .when(col("area_limpia") < 100, "Mediano")
                     .when(col("area_limpia") < 200, "Grande")
                     .otherwise("Muy Grande"))

# 4. Categoria de precio (en USD)
sdf = sdf.withColumn("precio_categoria",
                     when(col("precio_usd") < 500, "Económico")
                     .when(col("precio_usd") < 1000, "Moderado")
                     .when(col("precio_usd") < 2000, "Alto")
                     .otherwise("Lujo"))

print("Variables derivadas creadas")

Creando variables derivadas...
Variables derivadas creadas


In [35]:
# ============================================
# BLOQUE 7: EXTRACCION DE ZONA Y PUBLICADOR
# ============================================

# 1. Funcion para clasificar zona (CON MANEJO DE NULL)
def get_zona(distrito):
    if distrito is None:
        return "Otro"
    try:
        distrito_upper = distrito.upper().strip()
    except:
        return "Otro"

    norte = ["SAN MARTIN DE PORRES", "COMAS", "LOS OLIVOS", "INDEPENDENCIA", "CARABAYLLO", "PUENTE PIEDRA"]
    sur = ["CHORRILLOS", "BARRANCO", "SURQUILLO", "SAN JUAN DE MIRAFLORES", "VILLA MARIA DEL TRIUNFO",
           "VILLA EL SALVADOR", "PUNTA HERMOSA", "PUNTA NEGRA", "SAN BARTOLO"]
    este = ["ATE", "SAN JUAN DE LURIGANCHO", "LA MOLINA", "SANTIAGO DE SURCO", "SANTA ANITA", "LURIGANCHO"]
    oeste = ["SAN MIGUEL", "CALLAO", "BELLAVISTA", "LA PERLA"]
    centro = ["MIRAFLORES", "SAN ISIDRO", "LINCE", "JESUS MARIA", "MAGDALENA DEL MAR",
              "PUEBLO LIBRE", "BRENA", "CERCADO DE LIMA", "LA VICTORIA"]

    if distrito_upper in [d.upper() for d in norte]:
        return "Norte"
    elif distrito_upper in [d.upper() for d in sur]:
        return "Sur"
    elif distrito_upper in [d.upper() for d in este]:
        return "Este"
    elif distrito_upper in [d.upper() for d in oeste]:
        return "Oeste"
    elif distrito_upper in [d.upper() for d in centro]:
        return "Centro"
    else:
        return "Otro"

# Registrar UDF
from pyspark.sql.functions import udf
zona_udf = udf(get_zona, StringType())
sdf = sdf.withColumn("zona", zona_udf(col("distrito")))
print("Zona extraida")

# 2. Extraer publicador
sdf = sdf.withColumn("publicador",
                     when(col("fecha_publicacion").rlike("Publicado por"),
                          regexp_extract(col("fecha_publicacion"), "Publicado por (.*)", 1))
                     .otherwise(None))
print("Publicador extraido")

Zona extraida
Publicador extraido


In [36]:
# ============================================
# BLOQUE 8: SELECCIONAR Y GUARDAR DATASET FINAL
# ============================================

# 1. Seleccionar columnas finales
df_final = sdf.select(
    "titulo",
    "ubicacion",
    "distrito",
    "zona",
    "precio_usd",
    "precio_m2",
    "precio_categoria",
    "tipo_inmueble",
    "tipo_operacion",
    "dormitorios_limpio",
    "banos_limpio",
    "area_limpia",
    "tamano_categoria",
    "antiguedad",
    "anio_construccion",
    "mantenimiento",
    "publicador",
    "fecha_publicacion",
    "url"
)

# Renombrar columnas finales
df_final = df_final.withColumnRenamed("precio_usd", "precio") \
                   .withColumnRenamed("dormitorios_limpio", "dormitorios") \
                   .withColumnRenamed("banos_limpio", "banos") \
                   .withColumnRenamed("area_limpia", "area")

print(f"Columnas finales seleccionadas")
print(f"Total registros finales: {df_final.count()}")

# 2. Mostrar resultado final (con manejo de errores)
try:
    print("\nDataset final (muestra):")
    df_final.show(10, truncate=True)
except Exception as e:
    print(f"Error al mostrar: {e}")
    print("Mostrando con collect()...")
    for row in df_final.limit(5).collect():
        print(row)

# 3. Mostrar distribuciones
print("\nDistribucion por tipo de inmueble:")
df_final.filter(col("tipo_inmueble").isNotNull()) \
        .groupBy("tipo_inmueble").count().show()

print("\nDistribucion por zona:")
df_final.filter(col("zona").isNotNull()) \
        .groupBy("zona").count().show()

print("\nDistribucion por distrito (Top 10):")
df_final.filter(col("distrito").isNotNull()) \
        .groupBy("distrito").count() \
        .orderBy(col("count").desc()).show(10)

print("\nPrecio promedio por distrito (Top 5):")
from pyspark.sql.functions import avg
df_final.filter(col("distrito").isNotNull()) \
        .groupBy("distrito") \
        .agg(avg("precio").alias("precio_promedio")) \
        .orderBy(col("precio_promedio").desc()).show(5)

print("\nPrecio promedio por zona:")
df_final.filter(col("zona").isNotNull()) \
        .groupBy("zona") \
        .agg(avg("precio").alias("precio_promedio")) \
        .orderBy(col("precio_promedio").desc()).show()

# 4. Guardar SOLO en CSV
print("Convirtiendo a Pandas...")
df_pandas = df_final.toPandas()

csv_path = '/content/drive/MyDrive/lima_rent/dataset_clean.csv'
df_pandas.to_csv(csv_path, index=False)
print(f"Dataset guardado en CSV: {csv_path}")

# 5. Estadisticas finales
print("\nEstadisticas descriptivas:")
print(df_pandas[['precio', 'area', 'dormitorios', 'banos']].describe())

print("\nValores nulos por columna:")
print(df_pandas.isnull().sum())

print("\nDistribucion de precios por categoria:")
print(df_pandas['precio_categoria'].value_counts())

print("\nETL COMPLETADO EXITOSAMENTE")

Columnas finales seleccionadas
Total registros finales: 597

Dataset final (muestra):
+--------------------+--------------------+----------+------+------+------------------+----------------+-------------+--------------+-----------+-----+-----+----------------+----------+-----------------+-------------+--------------------+--------------------+--------------------+
|              titulo|           ubicacion|  distrito|  zona|precio|         precio_m2|precio_categoria|tipo_inmueble|tipo_operacion|dormitorios|banos| area|tamano_categoria|antiguedad|anio_construccion|mantenimiento|          publicador|   fecha_publicacion|                 url|
+--------------------+--------------------+----------+------+------+------------------+----------------+-------------+--------------+-----------+-----+-----+----------------+----------+-----------------+-------------+--------------------+--------------------+--------------------+
|Apartamento en al...|Ur. Santa Cruz, M...|Miraflores|Centro| 900.0| 8.